## Inflation Analysis 2024: A Comprehensive Study of US Price Dynamics

**We examine major US inflation measures -- CPI, Core CPI, PCE, Core PCE, 
and PPI -- alongside inflation expectations from the bond market (5-year 
breakeven) and consumer surveys (Michigan). The Fed Funds rate is overlaid 
to assess the monetary policy response.**

This notebook covers:

1. Data retrieval of key inflation and monetary policy series from FRED
2. Year-over-year and month-over-month annualized inflation rates
3. CPI vs PCE comparison (the Fed's preferred measure)
4. Inflation expectations vs realized inflation
5. Historical context: 1970s stagflation, 2008 crisis, COVID shock, 2022 peak
6. Visualization of inflation dynamics and Fed policy response

*Dependencies:*

- pandas, matplotlib, seaborn for data handling and visualization
- pandas\_datareader or fredapi for FRED data access
- FRED API key (set environment variable `FRED_API_KEY`)

*CHANGE LOG*

    2024-01-15  First version with comprehensive inflation analysis.

### Setup and imports

In [ ]:
#  Attempt to use fecon235 interface; fall back to direct FRED access.
try:
    from fecon235.fecon235 import *
    USE_FECON = True
    print("fecon235 loaded successfully.")
except Exception as e:
    USE_FECON = False
    print(f"fecon235 not available ({e}), using pandas_datareader for FRED.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

#  Configure plotting style consistent with repo notebooks
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

%matplotlib inline

### Data access helper

We define a unified data retrieval function. If `fecon235` is available, 
we use its `get()` function which handles FRED codes natively. 
Otherwise, we fall back to `pandas_datareader` or `fredapi`.

In [ ]:
def fetch_fred(code, start='1960-01-01'):
    """Retrieve FRED series, trying multiple backends."""
    if USE_FECON:
        try:
            return get(code)
        except Exception:
            pass
    #  Fallback: pandas_datareader
    try:
        import pandas_datareader.data as web
        df = web.DataReader(code, 'fred', start=start)
        df.columns = ['Y']
        return df
    except Exception:
        pass
    #  Fallback: fredapi
    try:
        import os
        from fredapi import Fred
        fred = Fred(api_key=os.environ.get('FRED_API_KEY', ''))
        s = fred.get_series(code, observation_start=start)
        return pd.DataFrame(s, columns=['Y'])
    except Exception as e:
        raise RuntimeError(f"Could not retrieve {code}: {e}")

print("Data retrieval function ready.")

## Data Collection

We retrieve the following series from FRED:

| Series | FRED Code | Description |
|:-------|:----------|:------------|
| CPI | CPIAUCSL | Consumer Price Index for All Urban Consumers |
| Core CPI | CPILFESL | CPI Less Food and Energy |
| PCE | PCEPI | Personal Consumption Expenditures Price Index |
| Core PCE | PCEPILFE | PCE Less Food and Energy |
| PPI | PPIACO | Producer Price Index, All Commodities |
| 5-Year Breakeven | T5YIE | 5-Year Breakeven Inflation Rate |
| Michigan Survey | MICH | University of Michigan Inflation Expectation |
| Fed Funds Rate | FEDFUNDS | Effective Federal Funds Rate |

In [ ]:
#  Price indices (monthly, seasonally adjusted)
cpi      = fetch_fred('CPIAUCSL')
core_cpi = fetch_fred('CPILFESL')
pce      = fetch_fred('PCEPI')
core_pce = fetch_fred('PCEPILFE')
ppi      = fetch_fred('PPIACO')

print("Price indices retrieved successfully.")
print(f"  CPI:      {cpi.index[0].strftime('%Y-%m')} to {cpi.index[-1].strftime('%Y-%m')}  ({len(cpi)} obs)")
print(f"  Core CPI: {core_cpi.index[0].strftime('%Y-%m')} to {core_cpi.index[-1].strftime('%Y-%m')}  ({len(core_cpi)} obs)")
print(f"  PCE:      {pce.index[0].strftime('%Y-%m')} to {pce.index[-1].strftime('%Y-%m')}  ({len(pce)} obs)")
print(f"  Core PCE: {core_pce.index[0].strftime('%Y-%m')} to {core_pce.index[-1].strftime('%Y-%m')}  ({len(core_pce)} obs)")
print(f"  PPI:      {ppi.index[0].strftime('%Y-%m')} to {ppi.index[-1].strftime('%Y-%m')}  ({len(ppi)} obs)")

In [ ]:
#  Inflation expectations and monetary policy (monthly)
bei5y    = fetch_fred('T5YIE',    start='2003-01-01')
mich     = fetch_fred('MICH',     start='1978-01-01')
fedfunds = fetch_fred('FEDFUNDS', start='1960-01-01')

print("Expectations and policy series retrieved.")
print(f"  5Y Breakeven:   {bei5y.index[0].strftime('%Y-%m')} to {bei5y.index[-1].strftime('%Y-%m')}  ({len(bei5y)} obs)")
print(f"  Michigan Survey: {mich.index[0].strftime('%Y-%m')} to {mich.index[-1].strftime('%Y-%m')}  ({len(mich)} obs)")
print(f"  Fed Funds Rate:  {fedfunds.index[0].strftime('%Y-%m')} to {fedfunds.index[-1].strftime('%Y-%m')}  ({len(fedfunds)} obs)")

### Data summary tables

In [ ]:
#  Combine latest values into a summary table
latest = pd.DataFrame({
    'Series': ['CPI', 'Core CPI', 'PCE', 'Core PCE', 'PPI',
               '5Y Breakeven', 'Michigan Survey', 'Fed Funds'],
    'FRED Code': ['CPIAUCSL', 'CPILFESL', 'PCEPI', 'PCEPILFE', 'PPIACO',
                  'T5YIE', 'MICH', 'FEDFUNDS'],
    'Latest Value': [
        cpi['Y'].iloc[-1], core_cpi['Y'].iloc[-1],
        pce['Y'].iloc[-1], core_pce['Y'].iloc[-1],
        ppi['Y'].iloc[-1], bei5y['Y'].iloc[-1],
        mich['Y'].iloc[-1], fedfunds['Y'].iloc[-1]
    ],
    'Latest Date': [
        cpi.index[-1].strftime('%Y-%m'), core_cpi.index[-1].strftime('%Y-%m'),
        pce.index[-1].strftime('%Y-%m'), core_pce.index[-1].strftime('%Y-%m'),
        ppi.index[-1].strftime('%Y-%m'), bei5y.index[-1].strftime('%Y-%m'),
        mich.index[-1].strftime('%Y-%m'), fedfunds.index[-1].strftime('%Y-%m')
    ]
})
print(latest.to_string(index=False))

## Year-over-Year Inflation Rates

The standard measure of inflation is the **year-over-year percentage change** 
in the price index. For a monthly price series $P_t$:

$$\text{YoY Inflation}_t = \left(\frac{P_t}{P_{t-12}} - 1\right) \times 100$$

This removes seasonal effects and gives a smooth reading of price changes.

In [ ]:
#  Compute year-over-year inflation rates (percent)
cpi_yoy      = (cpi['Y'] / cpi['Y'].shift(12) - 1) * 100
core_cpi_yoy = (core_cpi['Y'] / core_cpi['Y'].shift(12) - 1) * 100
pce_yoy      = (pce['Y'] / pce['Y'].shift(12) - 1) * 100
core_pce_yoy = (core_pce['Y'] / core_pce['Y'].shift(12) - 1) * 100
ppi_yoy      = (ppi['Y'] / ppi['Y'].shift(12) - 1) * 100

#  Drop NaN from the shift
cpi_yoy      = cpi_yoy.dropna()
core_cpi_yoy = core_cpi_yoy.dropna()
pce_yoy      = pce_yoy.dropna()
core_pce_yoy = core_pce_yoy.dropna()
ppi_yoy      = ppi_yoy.dropna()

#  Summary statistics
yoy_df = pd.DataFrame({
    'CPI': cpi_yoy, 'Core CPI': core_cpi_yoy,
    'PCE': pce_yoy, 'Core PCE': core_pce_yoy,
    'PPI': ppi_yoy
})

print("Year-over-Year Inflation Rates -- Descriptive Statistics")
print("=" * 65)
print(yoy_df.describe().round(2).to_string())

### Month-over-Month Annualized Inflation

Month-over-month changes, annualized, give a higher-frequency signal 
of inflation momentum. They are noisier than YoY but more timely:

$$\text{MoM Annualized}_t = \left(\left(\frac{P_t}{P_{t-1}}\right)^{12} - 1\right) \times 100$$

In [ ]:
#  Month-over-month annualized inflation rates
cpi_mom      = ((cpi['Y'] / cpi['Y'].shift(1)) ** 12 - 1) * 100
core_cpi_mom = ((core_cpi['Y'] / core_cpi['Y'].shift(1)) ** 12 - 1) * 100
pce_mom      = ((pce['Y'] / pce['Y'].shift(1)) ** 12 - 1) * 100
core_pce_mom = ((core_pce['Y'] / core_pce['Y'].shift(1)) ** 12 - 1) * 100

mom_df = pd.DataFrame({
    'CPI': cpi_mom, 'Core CPI': core_cpi_mom,
    'PCE': pce_mom, 'Core PCE': core_pce_mom
}).dropna()

#  Show recent 12 months
print("Month-over-Month Annualized Inflation (last 12 months)")
print("=" * 60)
print(mom_df.tail(12).round(2).to_string())

## CPI vs PCE: The Fed's Preferred Measure

> *"The FOMC focuses on PCE inflation in its quarterly economic 
> projections and also states its longer-run inflation goal in terms 
> of headline PCE."* -- James Bullard, Federal Reserve Bank of St. Louis

The CPI and PCE differ in three key ways:
1. **Substitution effect**: PCE weights adjust as consumers substitute goods; CPI uses fixed baskets
2. **Coverage**: PCE covers all consumption (including employer-paid healthcare); CPI covers out-of-pocket only
3. **Revisions**: PCE data are revised; CPI is not

Historically, CPI runs about 0.3 percentage points higher than PCE.

In [ ]:
#  CPI vs PCE spread
spread = cpi_yoy - pce_yoy
spread = spread.dropna()

fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

#  Panel 1: CPI and PCE YoY
ax1 = axes[0]
ax1.plot(cpi_yoy.index, cpi_yoy, label='CPI YoY', color='#2166ac', linewidth=1.2)
ax1.plot(pce_yoy.index, pce_yoy, label='PCE YoY', color='#b2182b', linewidth=1.2)
ax1.axhline(y=2.0, color='gray', linestyle='--', alpha=0.7, label='2% Target')
ax1.set_ylabel('Percent (%)')
ax1.set_title('CPI vs PCE: Year-over-Year Inflation')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

#  Panel 2: Spread (CPI - PCE)
ax2 = axes[1]
ax2.fill_between(spread.index, spread, 0, alpha=0.4, color='#4393c3')
ax2.axhline(y=spread.mean(), color='red', linestyle='--', alpha=0.7,
            label=f'Mean spread: {spread.mean():.2f}pp')
ax2.set_ylabel('Percentage Points')
ax2.set_title('CPI minus PCE Spread')
ax2.set_xlabel('Date')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAverage CPI-PCE spread: {spread.mean():.2f} percentage points")
print(f"Current spread:         {spread.iloc[-1]:.2f} percentage points")

### Core vs Headline: Stripping Out Volatility

Core inflation excludes food and energy prices, which are subject to 
supply shocks (oil embargoes, droughts, geopolitical disruptions). 
The Fed watches core measures for the underlying inflation trend, 
while headline captures what consumers actually experience.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

#  Panel 1: CPI headline vs core
ax1 = axes[0]
ax1.plot(cpi_yoy.index, cpi_yoy, label='CPI (Headline)', color='#2166ac', linewidth=1.0, alpha=0.8)
ax1.plot(core_cpi_yoy.index, core_cpi_yoy, label='Core CPI', color='#b2182b', linewidth=1.5)
ax1.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5)
ax1.set_ylabel('Percent (%)')
ax1.set_title('CPI: Headline vs Core')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

#  Panel 2: PCE headline vs core
ax2 = axes[1]
ax2.plot(pce_yoy.index, pce_yoy, label='PCE (Headline)', color='#2166ac', linewidth=1.0, alpha=0.8)
ax2.plot(core_pce_yoy.index, core_pce_yoy, label='Core PCE', color='#b2182b', linewidth=1.5)
ax2.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('Percent (%)')
ax2.set_title("PCE: Headline vs Core (Fed's Preferred Measure)")
ax2.set_xlabel('Date')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Inflation Expectations vs Realized Inflation

Inflation expectations are critical for monetary policy. The Fed monitors:

- **5-Year Breakeven Inflation (T5YIE)**: Market-implied expectations derived 
  from the spread between nominal Treasuries and TIPS. Available since 2003.
- **University of Michigan Consumer Survey (MICH)**: Survey-based 1-year ahead 
  expectations from consumers. Available since 1978.

Well-anchored expectations near the 2% target indicate credible monetary policy. 
De-anchoring -- as in the 1970s -- can create self-fulfilling inflation spirals.

In [ ]:
#  Align expectations with realized CPI YoY
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

#  Panel 1: Michigan survey vs realized CPI
ax1 = axes[0]
ax1.plot(mich.index, mich['Y'], label='Michigan 1Y Expectation',
         color='#e66101', linewidth=1.5)
ax1.plot(cpi_yoy.index, cpi_yoy, label='Realized CPI YoY',
         color='#5e3c99', linewidth=1.0, alpha=0.7)
ax1.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5, label='2% Target')
ax1.set_ylabel('Percent (%)')
ax1.set_title('Consumer Inflation Expectations vs Realized CPI')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(mich.index[0], cpi_yoy.index[-1])

#  Panel 2: 5Y Breakeven vs realized CPI (from 2003)
start_bei = '2003-01-01'
ax2 = axes[1]
ax2.plot(bei5y.index, bei5y['Y'], label='5Y Breakeven Inflation',
         color='#e66101', linewidth=1.5)
ax2.plot(cpi_yoy.loc[start_bei:].index, cpi_yoy.loc[start_bei:],
         label='Realized CPI YoY', color='#5e3c99', linewidth=1.0, alpha=0.7)
ax2.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5, label='2% Target')
ax2.set_ylabel('Percent (%)')
ax2.set_title('Market Inflation Expectations (5Y Breakeven) vs Realized CPI')
ax2.set_xlabel('Date')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Expectations vs Realized: Scatter Analysis

A scatter plot of inflation expectations against realized inflation 
reveals how well expectations track reality. Points on the 45-degree 
line indicate perfect foresight. Systematic deviations indicate bias.

In [ ]:
#  Scatter: Michigan expectations vs realized CPI (1-year later)
#  Shift CPI forward by 12 months to compare expectation with outcome
cpi_realized = cpi_yoy.shift(-12)  # what actually happened 12 months later
common_idx = mich.index.intersection(cpi_realized.dropna().index)

mich_vals = mich.loc[common_idx, 'Y'].values
real_vals = cpi_realized.loc[common_idx].values

fig, ax = plt.subplots(figsize=(8, 8))
scatter = ax.scatter(mich_vals, real_vals, alpha=0.3, s=20,
                     c=range(len(mich_vals)), cmap='viridis', edgecolors='none')
ax.plot([0, 16], [0, 16], 'r--', alpha=0.5, label='Perfect foresight (45\u00b0)')
ax.set_xlabel('Michigan Survey: Expected Inflation (%)')
ax.set_ylabel('Realized CPI YoY 12 Months Later (%)')
ax.set_title('Inflation Expectations vs Realized Inflation')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

cbar = plt.colorbar(scatter, ax=ax, label='Time (older \u2192 newer)')
cbar.set_ticks([])

plt.tight_layout()
plt.show()

corr = np.corrcoef(mich_vals, real_vals)[0, 1]
print(f"Correlation between Michigan expectations and realized CPI: {corr:.3f}")

## Historical Context: Key Inflationary Episodes

US inflation history contains several critical episodes that shaped 
monetary policy and economic thinking:

| Period | Event | Peak Inflation | Policy Response |
|:-------|:------|:---------------|:----------------|
| 1973-1975 | Oil embargo, wage-price spirals | CPI ~12% | Burns Fed: stop-go policy |
| 1979-1981 | Second oil shock, Volcker disinflation | CPI ~14.8% | Fed Funds to 20%, deep recession |
| 2008-2009 | Financial crisis, Great Recession | CPI briefly negative | ZIRP, QE1 |
| 2020-2021 | COVID supply shock, fiscal stimulus | Initial deflation scare | Emergency rate cuts, massive QE |
| 2021-2022 | Post-COVID surge, supply chains, Ukraine | CPI ~9.1% (June 2022) | Fastest hiking cycle since 1980s |

The annotations below mark these turning points on the inflation time series.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(cpi_yoy.index, cpi_yoy, color='#2166ac', linewidth=1.2, label='CPI YoY')
ax.plot(core_cpi_yoy.index, core_cpi_yoy, color='#b2182b', linewidth=1.0,
        alpha=0.7, label='Core CPI YoY')
ax.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5, label='2% Target')
ax.axhline(y=0.0, color='black', linewidth=0.5)

#  Annotate key historical episodes
annotations = [
    ('1974-12-01', 12.3, '1973-75\nOil Embargo'),
    ('1980-03-01', 14.8, '1980\nVolcker Peak'),
    ('2008-07-01', 5.5,  '2008\nFinancial Crisis'),
    ('2009-07-01', -2.1, '2009\nDeflation'),
    ('2020-05-01', 0.1,  '2020\nCOVID'),
    ('2022-06-01', 9.1,  '2022\nPost-COVID Peak'),
]

for date_str, y_pos, label in annotations:
    date = pd.Timestamp(date_str)
    ax.annotate(label, xy=(date, y_pos),
                xytext=(0, 25), textcoords='offset points',
                fontsize=9, ha='center', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.0),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.7))

#  Shade recession-like periods
shade_periods = [
    ('1973-11-01', '1975-03-01'),
    ('1980-01-01', '1980-07-01'),
    ('1981-07-01', '1982-11-01'),
    ('2007-12-01', '2009-06-01'),
    ('2020-02-01', '2020-04-01'),
]
for start, end in shade_periods:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
               alpha=0.1, color='gray')

ax.set_ylabel('Percent (%)')
ax.set_title('US Inflation: A Historical Perspective (1960-Present)')
ax.set_xlabel('Date')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Multi-Panel Time Series: All Inflation Measures

A comprehensive view of all four primary inflation measures -- CPI, Core CPI, 
PCE, and Core PCE -- shown simultaneously to reveal co-movement and divergences.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

measures = [
    (axes[0, 0], cpi_yoy,      'CPI YoY',      '#2166ac'),
    (axes[0, 1], core_cpi_yoy, 'Core CPI YoY', '#b2182b'),
    (axes[1, 0], pce_yoy,      'PCE YoY',      '#1b7837'),
    (axes[1, 1], core_pce_yoy, 'Core PCE YoY', '#762a83'),
]

for ax, series, title, color in measures:
    ax.plot(series.index, series, color=color, linewidth=1.0)
    ax.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=0.0, color='black', linewidth=0.3)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Percent (%)')
    ax.grid(True, alpha=0.3)
    #  Highlight recent period (2020+)
    recent = series.loc['2020-01-01':]
    if len(recent) > 0:
        ax.fill_between(recent.index, recent, 0, alpha=0.15, color=color)

axes[1, 0].set_xlabel('Date')
axes[1, 1].set_xlabel('Date')

fig.suptitle('US Inflation Measures: Year-over-Year', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Producer Price Index (PPI)

The PPI measures prices received by domestic producers. It is often viewed 
as a **leading indicator** for consumer inflation since increases in input 
costs eventually pass through to consumer prices.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(ppi_yoy.index, ppi_yoy, color='#d95f02', linewidth=1.0, alpha=0.8,
        label='PPI YoY')
ax.plot(cpi_yoy.index, cpi_yoy, color='#2166ac', linewidth=1.0, alpha=0.6,
        label='CPI YoY')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Percent (%)')
ax.set_title('Producer vs Consumer Price Inflation')
ax.set_xlabel('Date')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#  Correlation between PPI and CPI
common = pd.DataFrame({'PPI': ppi_yoy, 'CPI': cpi_yoy}).dropna()
print(f"PPI-CPI correlation (full sample): {common['PPI'].corr(common['CPI']):.3f}")

## Fed Funds Rate vs Inflation: The Policy Response

The Federal Reserve's primary tool for controlling inflation is the **Federal 
Funds rate** -- the overnight interbank lending rate. The **Taylor Rule** suggests 
the Fed should raise rates when inflation exceeds the 2% target.

Key FOMC decisions at turning points:
- **1979**: Volcker shifts to money supply targeting, rates spike to ~20%
- **2008**: Emergency cuts to near zero (ZIRP), beginning of QE era
- **2015-2018**: Gradual normalization ("dot plot" guidance)
- **2020 March**: Emergency cut to 0-0.25%, restart of QE
- **2022 March**: First hike of tightening cycle; fastest pace since 1980s
- **2023 July**: Final hike to 5.25-5.50%, holding at restrictive levels

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 7))

#  Inflation on left axis
color_infl = '#b2182b'
ax1.plot(cpi_yoy.index, cpi_yoy, color=color_infl, linewidth=1.2,
         label='CPI YoY', alpha=0.8)
ax1.plot(core_pce_yoy.index, core_pce_yoy, color='#762a83', linewidth=1.0,
         label='Core PCE YoY', alpha=0.7)
ax1.axhline(y=2.0, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Date')
ax1.set_ylabel('Inflation Rate (%)', color=color_infl)
ax1.tick_params(axis='y', labelcolor=color_infl)

#  Fed Funds on right axis
ax2 = ax1.twinx()
color_ff = '#2166ac'
ax2.plot(fedfunds.index, fedfunds['Y'], color=color_ff, linewidth=1.5,
         label='Fed Funds Rate', alpha=0.9)
ax2.set_ylabel('Fed Funds Rate (%)', color=color_ff)
ax2.tick_params(axis='y', labelcolor=color_ff)

#  Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

ax1.set_title('Inflation vs Federal Funds Rate: The Monetary Policy Response')
ax1.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

#  Real interest rate (Fed Funds - CPI YoY)
common_ff = pd.DataFrame({'FF': fedfunds['Y'], 'CPI': cpi_yoy}).dropna()
common_ff['Real_Rate'] = common_ff['FF'] - common_ff['CPI']
print(f"Current Fed Funds:     {fedfunds['Y'].iloc[-1]:.2f}%")
print(f"Current CPI YoY:       {cpi_yoy.iloc[-1]:.2f}%")
print(f"Real Fed Funds Rate:   {common_ff['Real_Rate'].iloc[-1]:.2f}%")

## Correlation Heatmap of Inflation Components

Understanding the co-movement between different inflation measures, 
expectations, and monetary policy helps gauge the coherence of the 
inflation regime.

In [ ]:
#  Build a combined DataFrame for correlation analysis
#  Use common date range where all series overlap
corr_df = pd.DataFrame({
    'CPI YoY': cpi_yoy,
    'Core CPI YoY': core_cpi_yoy,
    'PCE YoY': pce_yoy,
    'Core PCE YoY': core_pce_yoy,
    'PPI YoY': ppi_yoy,
    '5Y Breakeven': bei5y['Y'],
    'Michigan Exp': mich['Y'],
    'Fed Funds': fedfunds['Y'],
}).dropna()

print(f"Correlation matrix computed over {len(corr_df)} overlapping monthly observations")
print(f"  Date range: {corr_df.index[0].strftime('%Y-%m')} to {corr_df.index[-1].strftime('%Y-%m')}")

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool), k=1)
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, mask=mask,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation Heatmap: Inflation Measures, Expectations & Policy')

plt.tight_layout()
plt.show()

## The Post-COVID Inflation Episode (2020-2024)

The post-COVID period has been the most significant inflationary episode 
since the early 1980s. Key drivers included:

1. **Supply chain disruptions**: Semiconductor shortages, port congestion, 
   container shipping costs surging 10x
2. **Fiscal stimulus**: Multiple rounds totaling ~$5 trillion (CARES Act, 
   American Rescue Plan)
3. **Monetary accommodation**: Fed Funds at 0%, $120B/month asset purchases
4. **Energy shock**: Russia-Ukraine conflict (Feb 2022) spiked oil and 
   natural gas prices
5. **Labor market tightness**: "Great Resignation," wage-price pressures

The FOMC responded with the most aggressive tightening cycle since 
Paul Volcker, raising the Fed Funds rate from 0-0.25% to 5.25-5.50% 
between March 2022 and July 2023 (525 basis points in 16 months).

In [ ]:
#  Zoom into the post-COVID period
post_covid = '2019-06-01'

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

#  Panel 1: Headline and core inflation
ax1 = axes[0]
ax1.plot(cpi_yoy.loc[post_covid:].index, cpi_yoy.loc[post_covid:],
         label='CPI YoY', color='#2166ac', linewidth=1.5)
ax1.plot(core_cpi_yoy.loc[post_covid:].index, core_cpi_yoy.loc[post_covid:],
         label='Core CPI YoY', color='#b2182b', linewidth=1.5)
ax1.plot(core_pce_yoy.loc[post_covid:].index, core_pce_yoy.loc[post_covid:],
         label='Core PCE YoY', color='#762a83', linewidth=1.5, linestyle='--')
ax1.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5, label='2% Target')
ax1.set_ylabel('Percent (%)')
ax1.set_title('Post-COVID Inflation Surge and Disinflation')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)

#  Panel 2: Expectations
ax2 = axes[1]
ax2.plot(bei5y.loc[post_covid:].index, bei5y.loc[post_covid:]['Y'],
         label='5Y Breakeven', color='#e66101', linewidth=1.5)
ax2.plot(mich.loc[post_covid:].index, mich.loc[post_covid:]['Y'],
         label='Michigan Survey', color='#1b9e77', linewidth=1.5)
ax2.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('Percent (%)')
ax2.set_title('Inflation Expectations During the Episode')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)

#  Panel 3: Fed Funds response
ax3 = axes[2]
ax3.plot(fedfunds.loc[post_covid:].index, fedfunds.loc[post_covid:]['Y'],
         label='Fed Funds Rate', color='#2166ac', linewidth=2.0)
ax3.set_ylabel('Percent (%)')
ax3.set_title('FOMC Policy Response: Federal Funds Rate')
ax3.set_xlabel('Date')
ax3.legend(loc='upper left', fontsize=9)
ax3.grid(True, alpha=0.3)

#  Annotate key FOMC decisions
fomc_events = [
    ('2020-03-15', 'Emergency\nCut to 0%'),
    ('2022-03-16', 'First Hike\n+25bp'),
    ('2022-06-15', '+75bp\n(largest since 1994)'),
    ('2023-07-26', 'Final Hike\n5.25-5.50%'),
]
for date_str, label in fomc_events:
    date = pd.Timestamp(date_str)
    try:
        nearest = fedfunds.loc[post_covid:].index[
            fedfunds.loc[post_covid:].index.get_indexer([date], method='nearest')[0]
        ]
        y_val = fedfunds.loc[nearest, 'Y']
        ax3.annotate(label, xy=(nearest, y_val),
                     xytext=(0, 30), textcoords='offset points',
                     fontsize=8, ha='center',
                     arrowprops=dict(arrowstyle='->', color='black', lw=0.8),
                     bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.8))
    except Exception:
        pass

plt.tight_layout()
plt.show()

## Comparing Inflationary Episodes

How does the 2021-2023 inflation compare to previous episodes? 
We align each episode at its peak and track the disinflation path.

In [ ]:
#  Define inflationary episodes by approximate peak dates
episodes = {
    '1974 Oil Crisis':   ('1972-01-01', '1977-12-01', '1974-12-01'),
    '1980 Volcker':      ('1978-01-01', '1983-12-01', '1980-03-01'),
    '2008 Financial':    ('2007-01-01', '2010-12-01', '2008-07-01'),
    '2022 Post-COVID':   ('2020-01-01', None,         '2022-06-01'),
}

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

for (name, (start, end, peak)), color in zip(episodes.items(), colors):
    if end is None:
        episode = cpi_yoy.loc[start:]
    else:
        episode = cpi_yoy.loc[start:end]
    peak_date = pd.Timestamp(peak)
    #  Reindex relative to peak (in months)
    months_from_peak = ((episode.index - peak_date).days / 30.44).astype(int)
    ax.plot(months_from_peak, episode.values, label=name, color=color,
            linewidth=2.0 if '2022' in name else 1.2)

ax.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5, label='2% Target')
ax.axvline(x=0, color='black', linestyle=':', alpha=0.3, label='Peak')
ax.set_xlabel('Months from Peak')
ax.set_ylabel('CPI YoY (%)')
ax.set_title('Inflationary Episodes Aligned at Peak')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-36, 48)

plt.tight_layout()
plt.show()

## Summary and Current Readings

The table below presents the most recent readings across all series, 
providing a snapshot of the current inflation landscape.

In [ ]:
#  Final summary table
summary = pd.DataFrame({
    'Measure': [
        'CPI YoY', 'Core CPI YoY', 'PCE YoY', 'Core PCE YoY', 'PPI YoY',
        '5Y Breakeven', 'Michigan Survey', 'Fed Funds Rate',
        'CPI-PCE Spread', 'Real Fed Funds Rate'
    ],
    'Latest (%)': [
        round(cpi_yoy.iloc[-1], 2),
        round(core_cpi_yoy.iloc[-1], 2),
        round(pce_yoy.iloc[-1], 2),
        round(core_pce_yoy.iloc[-1], 2),
        round(ppi_yoy.iloc[-1], 2),
        round(bei5y['Y'].iloc[-1], 2),
        round(mich['Y'].iloc[-1], 2),
        round(fedfunds['Y'].iloc[-1], 2),
        round(cpi_yoy.iloc[-1] - pce_yoy.iloc[-1], 2),
        round(fedfunds['Y'].iloc[-1] - cpi_yoy.iloc[-1], 2),
    ],
    'Historical Mean (%)': [
        round(cpi_yoy.mean(), 2),
        round(core_cpi_yoy.mean(), 2),
        round(pce_yoy.mean(), 2),
        round(core_pce_yoy.mean(), 2),
        round(ppi_yoy.mean(), 2),
        round(bei5y['Y'].mean(), 2),
        round(mich['Y'].mean(), 2),
        round(fedfunds['Y'].mean(), 2),
        round((cpi_yoy - pce_yoy).dropna().mean(), 2),
        round((fedfunds['Y'].to_frame().set_axis(['val'], axis=1)
              .join(cpi_yoy.rename('cpi'), how='inner')
              .eval('val - cpi')).mean(), 2),
    ]
})

print("=" * 65)
print("INFLATION DASHBOARD: Current vs Historical")
print("=" * 65)
print(summary.to_string(index=False))
print("=" * 65)

## Conclusions

1. **The post-COVID inflation episode peaked in mid-2022** with CPI reaching 
   ~9.1% YoY, the highest since the early 1980s. Disinflation has been 
   significant but the "last mile" back to 2% has proven sticky, 
   particularly in services.

2. **Core PCE -- the Fed's preferred measure** -- consistently runs below 
   headline CPI. The spread between CPI and PCE reflects methodological 
   differences, primarily the substitution effect and broader coverage in PCE.

3. **Inflation expectations remained relatively well-anchored** during this 
   episode, unlike the 1970s when de-anchoring created a wage-price spiral. 
   The 5-year breakeven stayed in a 2-3% range even at the inflation peak, 
   suggesting market confidence in the Fed's eventual success.

4. **The FOMC's response was historically aggressive**: 525bp of hikes in 
   16 months. The resulting positive real interest rate indicates a restrictive 
   policy stance intended to cool demand and return inflation to target.

5. **PPI as a leading indicator** showed disinflation well before consumer 
   measures, consistent with the supply-chain normalization narrative.

---

*This notebook is designed for educational and research purposes. 
Data is sourced from FRED (Federal Reserve Economic Data) maintained 
by the Federal Reserve Bank of St. Louis.*